In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q21/q21_rewrite/checkpoints/pre_cell_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 4 ###

# Filter lineitems where receipt date > commit date
mask_date = lineitem.L_RECEIPTDATE > lineitem.L_COMMITDATE
fi = lineitem[mask_date]

# Compute unique supplier counts per order
orig_count = (
    lineitem
    .groupby('L_ORDERKEY', sort=False)
    .agg({'L_SUPPKEY': 'nunique'})
    .reset_index()
    .rename(columns={'L_SUPPKEY': 'orig_count'})
)
after_count = (
    fi
    .groupby('L_ORDERKEY', sort=False)
    .agg({'L_SUPPKEY': 'nunique'})
    .reset_index()
    .rename(columns={'L_SUPPKEY': 'after_count'})
)

# Identify valid orders (orig_count >1 and after_count ==1)
valid_orders = orig_count.merge(after_count, on='L_ORDERKEY', how='inner')
valid_orders = valid_orders[(valid_orders.orig_count > 1) & (valid_orders.after_count == 1)]['L_ORDERKEY']

# Restrict to orders with status 'F'
f_orders = orders[orders.O_ORDERSTATUS == 'F']['O_ORDERKEY']
valid_orders = valid_orders[valid_orders.isin(f_orders)]

# Pre-filter Saudi Arabian suppliers
supplier_sa = supplier.merge(
    nation[nation.N_NAME == 'SAUDI ARABIA'][['N_NATIONKEY']],
    left_on='S_NATIONKEY',
    right_on='N_NATIONKEY',
    how='inner'
)[['S_SUPPKEY', 'S_NAME']]

# Final aggregation: count waiting lineitems per Saudi supplier
total = (
    lineitem[mask_date & lineitem.L_ORDERKEY.isin(valid_orders)]
    .merge(supplier_sa, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')
    .groupby('S_NAME', as_index=False, sort=False)
    .size()
    .rename(columns={'size': 'NUMWAIT'})
    .sort_values(['NUMWAIT', 'S_NAME'], ascending=[False, True])
)